<a href="https://colab.research.google.com/github/anjaliii210/LLM-Post-Training-with-QLoRA-SFT-DPO/blob/main/2_sft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.2 MB/s  0:00:00


In [ ]:
import torch
import json
import os

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer, SFTConfig

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [ ]:
DATASET_NAME = "trl-lib/ultrafeedback_binarized"

dataset = load_dataset(
    DATASET_NAME,
    split="train"
)

print(dataset)
print(dataset.column_names)
print(dataset[0])

README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  131MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.14MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
    num_rows: 62135
})
['chosen', 'rejected', 'score_chosen', 'score_rejected']
{'chosen': [{'content': 'Use the pygame library to write a version of the classic game Snake, with a unique twist', 'role': 'user'}, {'content': "Sure, I'd be happy to help you write a version of the classic game Snake using the pygame library! Here's a basic outline of how we can approach this:\n\n1. First, we'll need to set up the game display and create a game object that we can use to handle the game's state.\n2. Next, we'll create the game's grid, which will be used to represent the game board. We'll need to define the size of the grid and the spaces within it.\n3. After that, we'll create the snake object, which will be used to represent the player's movement. We'll need to define the size of the snake and the speed at which it moves.\n4. We'll also need to create a food object, which will be used to represent the food t

In [ ]:
dataset = dataset.shuffle(seed=42).select(range(1000))

train_raw = dataset.select(range(800))
eval_raw = dataset.select(range(800, 900))
test_raw = dataset.select(range(900, 1000))

print(len(train_raw), len(eval_raw), len(test_raw))

800 100 100


In [ ]:
#responses converted to sft format
def format_sft(example):

    chosen = example["chosen"]

    prompt = chosen[:-1]
    completion = chosen[-1:]

    assert completion[0]["role"] == "assistant"

    return {
        "prompt": prompt,
        "completion": completion
    }


sft_train = train_raw.map(
    format_sft,
    remove_columns=train_raw.column_names
)

sft_eval = eval_raw.map(
    format_sft,
    remove_columns=eval_raw.column_names
)

sft_test = test_raw.map(
    format_sft,
    remove_columns=test_raw.column_names
)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
print("PROMPT:")
print(sft_train[0]["prompt"])

print("\nCOMPLETION:")
print(sft_train[0]["completion"])

PROMPT:
[{'content': 'INPUT ARTICLE: Article: Different laptop models have different numeric pad configurations.  If your U, I, and O keys have 4, 5, and 6 printed in the lower corner, you have an older laptop with an alternate numeric pad. See the next section for details on using it. The ThinkPad line of laptops do not use an alternate numeric pad. You\'ll need to use the method in this section as a workaround. Some larger models have a dedicated numeric pad. Click the "Start" button in the lower-right corner of the desktop. In many versions of Windows, this is just a Windows icon. The Start menu will appear above the button. If you are using Windows 8 and don\'t see the Start button, press ⊞ Win on the keyboard. This will open the Start screen. You can start typing immediately when the Start menu or screen is open to begin searching. You\'ll see "On-Screen Keyboard" in the search results. If your Lenovo laptop does not have a NumLock key, you can use the Windows On-Screen Keyboard t

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [ ]:
formatted = tokenizer.apply_chat_template(
    sft_train[0]["prompt"] + sft_train[0]["completion"],
    tokenize=False
)

print(formatted)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
INPUT ARTICLE: Article: Different laptop models have different numeric pad configurations.  If your U, I, and O keys have 4, 5, and 6 printed in the lower corner, you have an older laptop with an alternate numeric pad. See the next section for details on using it. The ThinkPad line of laptops do not use an alternate numeric pad. You'll need to use the method in this section as a workaround. Some larger models have a dedicated numeric pad. Click the "Start" button in the lower-right corner of the desktop. In many versions of Windows, this is just a Windows icon. The Start menu will appear above the button. If you are using Windows 8 and don't see the Start button, press ⊞ Win on the keyboard. This will open the Start screen. You can start typing immediately when the Start menu or screen is open to begin searching. You'll see "On-Screen Keyboard" in the search results. If yo

In [ ]:
encoded = tokenizer(
    formatted,
    return_tensors="pt"
)

print("Token IDs:", encoded["input_ids"])
print("Sequence length:", encoded["input_ids"].shape)

Token IDs: tensor([[151644,   8948,    198,  ...,      4, 151645,    198]])
Sequence length: torch.Size([1, 1318])


Fresh QLoRA model

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 37.1 MB/s  0:00:01


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

model.config.use_cache = False

print("Base model loaded")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded


In [ ]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

In [ ]:
#attaching lora with same config as in nb 2
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


SFT Training

In [ ]:
sft_args = SFTConfig(
    output_dir="./sft_checkpoints",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    lr_scheduler_type="cosine",

    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optim="paged_adamw_8bit",

    max_length=512,
    completion_only_loss=True,
    packing=False,

    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,

    report_to="none"
)

In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

print("Adapters converted to FP32")

Adapters converted to FP32


In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype)

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.1.self_attn.k_proj.lora_B.default.w

In [ ]:
for name, p in model.named_parameters():
    if p.requires_grad:
        print(name, p.dtype)

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.1.self_attn.k_proj.lora_B.default.w

In [ ]:
for name, p in model.named_parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

In [ ]:
sft_args.fp16 = True
sft_args.bf16 = False

print("FP16:", sft_args.fp16)
print("BF16:", sft_args.bf16)

FP16: True
BF16: False


In [ ]:
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=sft_train,
    eval_dataset=sft_eval,
    processing_class=tokenizer
)

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
# Ensure all trainable adapter parameters are strictly float32 to prevent BFloat16 gradients
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
        if param.grad is not None:
            param.grad.data = param.grad.data.to(torch.float32)

training_result = trainer.train()
print(training_result)

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.319113,1.340931,1.305618,0.668465,254390.000000


TrainOutput(global_step=93, training_loss=1.3862911757602487, metrics={'train_runtime': 363.3391, 'train_samples_per_second': 2.031, 'train_steps_per_second': 0.256, 'total_flos': 549575201472000.0, 'train_loss': 1.3862911757602487, 'epoch': 1.0})


In [ ]:
eval_results = trainer.evaluate()

print("Evaluation loss:", eval_results["eval_loss"])

Training Loss,Validation Loss,Epoch,Entropy,Mean Token Accuracy,Num Tokens
1.319113,1.340931,1,1.305618,0.668465,254390.000000


Evaluation loss: 1.3409309387207031


In [ ]:
import math
perplexity = math.exp(eval_results["eval_loss"])

print("Perplexity:", perplexity)

Perplexity: 3.8226004562362923


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SFT_PATH = "/content/drive/MyDrive/LLM_PostTraining/sft_adapter"

trainer.save_model(SFT_PATH)
trainer.model.save_pretrained(SFT_PATH)
tokenizer.save_pretrained(SFT_PATH)

print("Trained SFT adapter saved!")

Mounted at /content/drive
Trained SFT adapter saved!


In [ ]:
!find /content/drive/MyDrive -name "adapter_config.json"

/content/drive/MyDrive/LLM_PostTraining/qlora_adapter/adapter_config.json
/content/drive/MyDrive/LLM_PostTraining/sft_adapter/adapter_config.json
